In [1]:
!pip install ebooklib beautifulsoup4


In [ ]:
import gradio as gr
import os
import shutil
import traceback
from collections import defaultdict
from ebooklib import epub
from bs4 import BeautifulSoup
from ebooklib.epub import EpubHtml
from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chains.question_answering import load_qa_chain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter

CHROMA_DIR = "chroma_db"

# EPUB loader
def load_epub(file_path):
    book = epub.read_epub(file_path)
    text = ""
    for item in book.get_items():
        if isinstance(item, EpubHtml):
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            text += soup.get_text(separator="\n")
    if not text.strip():
        raise ValueError(f"❌ {file_path} no contiene texto legible.")
    return [Document(page_content=text, metadata={"source_doc": os.path.basename(file_path)})]

# General loader
def process_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return PyPDFLoader(file_path).load()
    elif ext == ".txt":
        return TextLoader(file_path).load()
    elif ext == ".docx":
        return Docx2txtLoader(file_path).load()
    elif ext == ".epub":
        return load_epub(file_path)
    return []

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# LM Studio model
def init_llm():
    return ChatOpenAI(
        openai_api_key="lm-studio",
        openai_api_base="http://localhost:1234/v1",
        model="meta-llama-3.1-8b-instruct"
    )

# Indexar
def index_documents(files):
    if not files:
        return "❌ No se subieron archivos.", [], ""

    all_docs = []
    doc_names = []
    log = ""

    for file in files:
        name = os.path.basename(file.name)
        try:
            docs = process_file(file.name)
            chunks = splitter.split_documents(docs)
            for chunk in chunks:
                chunk.metadata["source_doc"] = name
            all_docs.extend(chunks)
            doc_names.append(name)
            log += f"✅ {name} - {len(chunks)} chunks\n"
        except Exception as e:
            log += f"❌ {name} - Error: {str(e)}\n"

    if all_docs:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
        db.add_documents(all_docs)
        return f"📁 Documentos procesados: {', '.join(doc_names)}", doc_names, log
    else:
        return "❌ No se indexaron documentos válidos.", [], log


# Reset base
def reset_chroma():
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)
    return "🧹 Base de datos eliminada.", [], ""

def load_existing_documents():
    if not os.path.exists(CHROMA_DIR):
        return [], "ℹ️ No hay base previa de documentos."

    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

        # Extraer documentos únicos por metadata
        docs = db.get()['metadatas']
        doc_names = list({meta.get("source_doc", "desconocido") for meta in docs})
        return doc_names, f"📂 Base cargada con {len(doc_names)} documentos."
    except Exception as e:
        return [], f"❌ Error al cargar la base existente: {str(e)}"


# Reformular
def generate_variations(query):
    llm = init_llm()
    prompt = (
        f"Genera 5 variaciones de esta pregunta para mejorar la recuperación semántica, sin cambiar el idioma:\n"
        f"Pregunta original: {query}\n"
        f"Variaciones:"
    )
    response = llm.predict(prompt)
    variations = [q.strip("- ") for q in response.split("\n") if q.strip()]
    return variations[:5] if len(variations) >= 5 else variations

# Agrupar chunks por documento para vista previa
def preview_chunks(query, selected_docs, k):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

        search_kwargs = {"k": k}
        if selected_docs:
            search_kwargs["filter"] = {"source_doc": {"$in": selected_docs}}

        retriever = db.as_retriever(search_kwargs=search_kwargs)
        docs = retriever.get_relevant_documents(query)

        grouped = defaultdict(list)
        for i, doc in enumerate(docs):
            name = doc.metadata.get("source_doc", "desconocido")
            grouped[name].append((i+1, doc.page_content[:500].replace('\n', ' ') + "..."))

        output = ""
        for doc_name, chunks in grouped.items():
            output += f"📄 {doc_name}:\n"
            for idx, chunk in chunks:
                output += f"   🔸 Chunk {idx}: {chunk}\n"
            output += "\n"

        return output.strip()

    except Exception as e:
        return f"❌ Error al previsualizar chunks: {str(e)}"
    
def wrap_generate_variations(query):
    if not query.strip():
        return "⚠️ Ingresa una pregunta primero."
    variations = generate_variations(query)
    return "\n".join([f"- {v}" for v in variations])


# Consultar
def answer_query(query, selected_docs, show_chunks, show_logs, idioma, k, system_instruction):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
        llm = init_llm()

        logs, chunks_text = "", ""
        variations = generate_variations(query)
        logs += f"🔁 Reformulaciones:\n" + "\n".join([f"- {v}" for v in variations]) + "\n\n"

        search_kwargs = {"k": k}
        if selected_docs:
            search_kwargs["filter"] = {"source_doc": {"$in": selected_docs}}

        retriever = db.as_retriever(search_kwargs=search_kwargs)
        docs = retriever.get_relevant_documents(query)

        if show_logs:
            logs += f"🔍 Consulta: {query}\n📄 Documentos: {', '.join(selected_docs) if selected_docs else 'Todos'}\n"

        if show_chunks:
            chunks_text = preview_chunks(query, selected_docs, k)

        idioma_map = {
            "Español": "Responde en español: ",
            "Inglés": "Answer in English: ",
            "Idioma original": ""
        }
        prefijo = idioma_map.get(idioma, "")

        custom_prompt = PromptTemplate.from_template(
    """{idioma_instruccion}
{system_instruction}

Responde la siguiente pregunta únicamente usando la información provista en los fragmentos.
Si no sabes la respuesta con certeza, responde: "No tengo suficiente información".

Fragmentos:
{context}

Pregunta: {question}
Respuesta:"""
)


        chain = load_qa_chain(llm=llm, chain_type="stuff", prompt=custom_prompt)
        answer = chain.run(
        input_documents=docs,
        question=query,
        idioma_instruccion=prefijo,
        system_instruction=system_instruction or ""
)


        return answer, chunks_text, logs

    except Exception as e:
        return f"❌ Error: {str(e)}", "", traceback.format_exc()

# Interfaz
initial_docs, initial_status = load_existing_documents()

with gr.Blocks() as demo:
    gr.Markdown("# 📚 RAG con múltiples documentos + LM Studio + Chroma")

    file_input = gr.File(file_types=[".pdf", ".docx", ".txt", ".epub"], file_count="multiple")
    upload_btn = gr.Button("📥 Cargar documentos")
    reset_btn = gr.Button("🧹 Reiniciar base de datos")
    doc_checkboxes = gr.CheckboxGroup(choices=initial_docs, value=initial_docs, label="Selecciona documentos para consultar", interactive=True)
    status = gr.Textbox(label="📦 Estado", value=initial_status)


    gr.Markdown("---")

    question = gr.Textbox(label="❓ Pregunta")
    generate_btn = gr.Button("🔁 Generar reformulaciones")
    variations_box = gr.Textbox(label="📓 Reformulaciones sugeridas", lines=6)
    idioma = gr.Radio(choices=["Español", "Inglés", "Idioma original"], value="Español", label="🌐 Idioma de respuesta")
    chunk_k = gr.Slider(minimum=1, maximum=20, step=1, value=5, label="🔢 Nº de chunks relevantes (k)")
    system_prompt = gr.Textbox(
    label="🧠 Instrucción del sistema (opcional)",
    lines=2,
    placeholder="Ej: Sé breve, directo y usa lenguaje técnico..."
)
    btn_ask = gr.Button("🤖 Preguntar")
    answer = gr.Textbox(label="💬 Respuesta", lines=4)
 

    with gr.Tabs():
        with gr.Tab("📚 Chunks relevantes"):
            chunk_text = gr.Textbox(label="", lines=12, visible=True)
        with gr.Tab("📝 Log del sistema"):
            logs_box = gr.Textbox(label="", lines=10, visible=True)

    show_chunks = gr.Checkbox(label="🔍 Ver chunks", value=True)
    show_logs = gr.Checkbox(label="🛠️ Ver logs", value=True)

    def wrap_index(files):
        msg, docs, log = index_documents(files)
        return msg, gr.update(choices=docs, value=docs), log

    def wrap_reset():
        msg, empty, log = reset_chroma()
        return msg, gr.update(choices=[]), log

    upload_btn.click(wrap_index, inputs=file_input, outputs=[status, doc_checkboxes, logs_box])
    reset_btn.click(wrap_reset, inputs=None, outputs=[status, doc_checkboxes, logs_box])
    generate_btn.click(wrap_generate_variations, inputs=question, outputs=variations_box)
    btn_ask.click(answer_query, inputs=[question, doc_checkboxes, show_chunks, show_logs, idioma, chunk_k], outputs=[answer, chunk_text, logs_box])
    btn_ask.click(
    answer_query,
    inputs=[question, doc_checkboxes, show_chunks, show_logs, idioma, chunk_k, system_prompt],
    outputs=[answer, chunk_text, logs_box]
)


    # Auto-previsualización de chunks
    question.change(preview_chunks, inputs=[question, doc_checkboxes, chunk_k], outputs=chunk_text)
    doc_checkboxes.change(preview_chunks, inputs=[question, doc_checkboxes, chunk_k], outputs=chunk_text)
    chunk_k.change(preview_chunks, inputs=[question, doc_checkboxes, chunk_k], outputs=chunk_text)

    demo.queue()

demo.launch()


C:\Users\jaime\AppData\Local\Temp\ipykernel_22484\1597664902.py:98: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
C:\Users\jaime\AppData\Local\Temp\ipykernel_22484\1597664902.py:99: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=CHROMA_DIR, embedding

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\jaime\AppData\Local\Temp\ipykernel_22484\1597664902.py:132: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)
c:\Users\jaime\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\jaime\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\jaime\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forw